# 03_retrieval_grader

04_retrieval_grader.py — Retrieval Grading (검색 문서 채점)

벡터 검색으로 top-k 를 가져온 뒤, 각 문서를 LLM 으로 yes/no 채점.
임베딩 유사도 점수와는 별개로 "의미상 정말 답에 도움 되는가" 를 한 번 더 거른다.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '03_retrieval_grader.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
04_retrieval_grader.py — Retrieval Grading (검색 문서 채점)

벡터 검색으로 top-k 를 가져온 뒤, 각 문서를 LLM 으로 yes/no 채점.
임베딩 유사도 점수와는 별개로 "의미상 정말 답에 도움 되는가" 를 한 번 더 거른다.
"""
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))

from _common import get_llm, binary_yesno, get_vectorstore, SAMPLE_DOCS, banner, llm_unavailable


GRADE_SYSTEM = (
    "너는 검색된 문서의 관련성을 평가하는 채점자다. "
    "엄격할 필요는 없다 — 문서에 질문과 관련된 키워드나 의미가 있으면 yes."
)


def main() -> None:
    llm = get_llm()
    if llm is None:
        llm_unavailable()
        return

    vs = get_vectorstore(SAMPLE_DOCS)
    retriever = vs.as_retriever(search_kwargs={"k": 5})

    question = "Self-RAG 의 반성 토큰은 어떤 종류가 있나?"
    docs = retriever.invoke(question)

    banner(f"Retrieval Grading — query: {question!r}")
    kept = []
    for d in docs:
        doc_id = d.metadata.get("id")
        topic = d.metadata.get("topic")
        score = binary_yesno(
            llm,
            GRADE_SYSTEM,
            f"검색된 문서:\n\n{d.page_content}\n\n사용자 질문: {question}",
        )
        flag = "✅ keep" if score == "yes" else "❌ drop"
        print(f"  {flag} [{doc_id} / {topic}]  {d.page_content[:60]}…")
        if score == "yes":
            kept.append(d)

    print(f"\n  → 통과 {len(kept)}/{len(docs)} 개 (관련 없는 문서는 LLM 컨텍스트에서 제외됨)")


if __name__ == "__main__":
    main()

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7207.52it/s]


📌 Retrieval Grading — query: 'Self-RAG 의 반성 토큰은 어떤 종류가 있나?'


  ✅ keep [doc07 / self_rag]  Self-RAG 는 모델이 생성 도중 특수한 반성 토큰(Retrieve / IsRelevant / IsSup…


  ❌ drop [doc05 / naive_rag]  Naive RAG 는 질문을 임베딩해 벡터DB에서 top-k 청크를 검색하고 그대로 LLM 에 넣는 가장 단…


  ❌ drop [doc06 / advanced_rag]  Advanced RAG 는 검색 전·후를 최적화한다. 사전 처리에는 의미 단위 청킹, 쿼리 재작성, Mult…


  ❌ drop [doc09 / adaptive_rag]  Adaptive RAG 는 질문 복잡도를 먼저 분류해 그에 맞는 최소한의 경로를 선택한다. 단순 질문은 LL…


  ❌ drop [doc08 / crag]  Corrective RAG(CRAG) 는 벡터 검색 결과의 신뢰도를 경량 평가기로 매긴 뒤 세 갈래로 분기한…

  → 통과 1/5 개 (관련 없는 문서는 LLM 컨텍스트에서 제외됨)
